In [ ]:
# ** 랭체인 프레임워크 제공 모듈로 PDF 문서 읽기 **
# langchain_community 라이브러리의 document_loaders 모듈에는 PDF 문서에서 텍스트를 추출하여 파일 내용을 Document 객체로 변환하는 다양한 유형의 Document Loader를 제공한다.

# 방법1) PyPDFLoader 이용하여 PDF 파일 데이터 가져오기 -----------------------
# PyPDFLoader : 랭체인 문서 로더 중 하나로 PDF파일을 로드하여 페이지 단위로 텍스트를 추출하고, 이를 랭체인의 Document 객체 리스트로 반환하는 역할을 함
# langchain_community가 제공하는 PyPDFLoader를 사용하여 PDF 파일에서 텍스트를 추출한다.
# PyPDFLoader 인스턴스를 사용하여 특정 PDF 파일을 열고, len(pages)를 통해 변환된 Document 객체의 개수를 얻는다.

!pip install pypdf langchain_community
!pip install openai python-dotenv


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

pdf_filepath = 'sample.pdf'

loader = PyPDFLoader(pdf_filepath)
pages = loader.load()

print(type(loader), type(pages))
print(len(pages))

print(pages[1].page_content)
print(pages[1].metadata)

In [ ]:
print("각 페이지 요약")

for i, page in enumerate(pages, start=1):
    prompt = f"다음 PDF 페이지 내용을 2줄로 요약:\n\n{page.page_content}"

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    print(f"페이지 {i} 요약 ---")
    print(response.output_text)

print("\n전체 문서 요약:")

full_text = "\n".join(p.page_content for p in pages)

prompt = "다음 문서를 10줄로 자연스러운 서술형 문단으로 요약해줘. :\n" + full_text

response_all = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response_all.output_text)


In [ ]:
"""
방법2) 형식이 없는 PDF 문서: UnstructuredPDFLoader 이용하여 PDF 파일 데이터 가져오기 ---
UnstructuredPDFLoader 클래스를 사용하여 PDF 파일에서 텍스트를 추출할 때는 내부적으로 unstructured 라이브러리의 기능을 활용한다. unstructured 라이브러리는 PDF 파일 내의 다양한 텍스트 조각(chunk)를 서로 다른 "elements"로 생성하고, 별도 설정을 하지 않으면 기본적으로 이러한 elements를 결합하여 하나의 텍스트 데이터로 반환한다.
즉, 각 페이지 또는 의미 있는 chunk롤 나눠 document 객체로 반환함. 메타데이터가 추가되어 페이지번호, 파일 경로 등의 정보를 각 문서에 포함한다.
"""
!apt-get update   # 패키지 목록을 최신 상태로 갱신
# Colab 리눅스 환경에 PDF 처리와 OCR 관련 프로그램을 설치
# poppler-utils : PDF 파일을 분석하고 처리하는 도구 모음
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-kor
!pip install -U "unstructured[pdf]" langchain_community

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader
import os

pdf_filepath = "sample.pdf"

if not os.path.exists(pdf_filepath):
    raise FileNotFoundError(f"{pdf_filepath} 파일을 찾을 수 없습니다.")

loader_single = UnstructuredPDFLoader(
    pdf_filepath,
    mode="single",
    languages=["ko"]
)

docs_single = loader_single.load()
print("single mode 결과")
print("문서 개수:", len(docs_single))

if docs_single:
    print("metadata:", docs_single[0].metadata)
    print("내용 일부:", docs_single[0].page_content[:100])

print("----------")

In [ ]:
"""
방법3) PDF 문서의 메타 데이터를 상세하게 추출 하기(PyMuPDFLoader 이용) -----------------
PyMuPDFLoader 클래스는 PyMuPDF를 사용하여 PDF 파일의 페이지를 로드하고, 각 페이지를 개별 document 객체로 추출한다. 특히 PDF 문서의 자세한 메타데이터를 추출하는 데 강점이 있다.

설치 : !pip install pymupdf
PyMuPDFLoader 인스턴스를 사용하여 지정된 PDF 파일을 로드하면, 각 페이지가 하나의 Document 객체로 일대일로 변환된다. len(pages)를 통해 파일 내 페이지 수를 계산하면 n개의 문서 객체로 변환되어 있다.
"""
!pip install pymupdf

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
pdf_filepath = 'sample.pdf'
loader = PyMuPDFLoader(pdf_filepath)
pages = loader.load()
print(len(pages))
print(pages[0].page_content)  # 첫번째 Document 객체의 내용(page_content)을 출력
# 이번에는 첫번째 Document 객체의 metadata를 출력해 본다. 이전 로더들과 비교해 더 상세한 정보를 담고 있다.
print(pages[0].metadata)  # {'producer': 'ezPDF Builder 2006', 'creator': 'ezPDF Builder 2006', ...

import json
print(json.dumps(pages[0].metadata, indent=2, ensure_ascii=False))   # metadata의 전체 구조 확인 가능
print(pages[0].metadata["producer"])   # producer 만 출력하기



In [ ]:
"""
방법4) 온라인 PDF 문서 로드 (OnlinePDFLoader 이용) -----------------------
OnlinePDFLoader는 LangChain에서 제공하는 문서 로더 중 하나로, 웹에 공개된 PDF 파일(URL) 을 직접 다운 로드하여 로드할 수 있게 해주는 도구다.
실습에서는 "Transformers" 논문(https://arxiv.org/pdf/1706.03762.pdf)을 로드하여 페이지 내용을 추출하고, 로드된 페이지 수와 첫 페이지의 내용 일부를 출력하는 과정을 처리한다.
"""

from langchain_community.document_loaders import OnlinePDFLoader
# Transformers 논문을 로드
loader = OnlinePDFLoader("https://arxiv.org/pdf/1706.03762.pdf")
pages = loader.load()
print(len(pages))   # 1
# 로드된 문서 객체의 내용을 출력하여 확인. 첫 페이지의 텍스트 내용 중 처음 1000자를 출력.
print(pages[0].page_content[:1000])
